# GRU Model Training for WESAD Dataset

This notebook trains a GRU model to classify stress, amusement, and baseline states using the WESAD dataset.

In [1]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import glob

## 1. Configuration and Setup

In [2]:
# Configuration
DATASET_PATH = '../../Dataset/WESAD'
TARGET_USERS = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
TARGET_LABELS = {1: 0, 2: 1, 3: 2} # 1: baseline, 2: stress, 3: amusement -> 0, 1, 2
N_FEATURES = 6 # ACC (3), BVP (1), EDA (1), TEMP (1)
DOWNSAMPLE_RATE = 4 # Hz, the rate to downsample all signals to
WINDOW_SIZE_SEC = 60 # seconds
STRIDE_SEC = 1 # seconds

# GRU Hyperparameters
HIDDEN_DIM = 128
LAYER_DIM = 2
OUTPUT_DIM = len(TARGET_LABELS)
BATCH_SIZE = 64
NUM_EPOCHS = 20
LEARNING_RATE = 0.001

# Setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


c:\Users\user\.conda\envs\myenv\lib\site-packages\torch\cuda\__init__.py:182: UserWarning: cudaGetDeviceCount() returned cudaErrorNotSupported, likely using older driver or on CPU machine (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:88.)
  return torch._C._cuda_getDeviceCount() > 0


## 2. Data Loading and Preprocessing

In [3]:
def load_and_preprocess_data(subject_path):
    """Loads a single subject's data, downsamples, and synchronizes."""
    with open(subject_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    # --- Extract Wrist Data --- #
    wrist_data = data['signal']['wrist']
    acc = wrist_data['ACC']
    bvp = wrist_data['BVP']
    eda = wrist_data['EDA']
    temp = wrist_data['TEMP']
    labels = data['label']

    # --- Downsample --- #
    # Original sampling rates: ACC=32Hz, BVP=64Hz, EDA=4Hz, TEMP=4Hz, label=700Hz
    acc_down = acc[::32 // DOWNSAMPLE_RATE]
    bvp_down = bvp[::64 // DOWNSAMPLE_RATE]
    eda_down = eda[::4 // DOWNSAMPLE_RATE]
    temp_down = temp[::4 // DOWNSAMPLE_RATE]
    
    # Align labels by resampling at the data points' timestamps
    label_timestamps = np.arange(0, len(labels) / 700.0, 1/700.0)
    data_timestamps = np.arange(0, len(acc_down) / DOWNSAMPLE_RATE, 1/DOWNSAMPLE_RATE)
    idx = np.searchsorted(label_timestamps, data_timestamps, side='left')
    idx = np.clip(idx, 0, len(labels) - 1)
    labels_down = labels[idx].astype(int)
    
    # Find the minimum length to truncate all signals
    min_len = min(len(acc_down), len(bvp_down), len(eda_down), len(temp_down), len(labels_down))
    
    # --- Combine Features --- #
    # Shape: (min_len, N_FEATURES)
    features = np.concatenate([
        acc_down[:min_len],
        bvp_down[:min_len],
        eda_down[:min_len],
        temp_down[:min_len]
    ], axis=1)
    
    labels_final = labels_down[:min_len]
    
    return features, labels_final

def create_windows(features, labels):
    """Creates sliding windows of data and corresponding labels."""
    window_samples = WINDOW_SIZE_SEC * DOWNSAMPLE_RATE
    stride_samples = STRIDE_SEC * DOWNSAMPLE_RATE

    X, y = [], []
    for i in range(0, len(features) - window_samples, stride_samples):
        window_features = features[i : i + window_samples]
        window_labels = labels[i : i + window_samples]
        
        # Use the most frequent label in the window as the segment's label
        most_frequent_label = np.bincount(window_labels).argmax()
        
        if most_frequent_label in TARGET_LABELS:
            X.append(window_features)
            y.append(TARGET_LABELS[most_frequent_label])

    return np.array(X), np.array(y)

In [4]:
# --- Process all subjects --- #
all_X, all_y = [], []
for user in TARGET_USERS:
    subject_path = os.path.join(DATASET_PATH, user, f'{user}.pkl')
    if os.path.exists(subject_path):
        print(f'Processing {user}...')
        features, labels = load_and_preprocess_data(subject_path)
        X, y = create_windows(features, labels)
        all_X.append(X)
        all_y.append(y)

X_combined = np.concatenate(all_X, axis=0)
y_combined = np.concatenate(all_y, axis=0)

print(f'\nTotal windows created: {len(X_combined)}')
print(f'Feature shape: {X_combined.shape}')
print(f'Label distribution: {np.bincount(y_combined)}')

# --- Split Data FIRST (Data Leakage 방지) --- #
# Train/Val/Test = 70% / 15% / 15%
X_train_unshaped, X_temp_unshaped, y_train, y_temp = train_test_split(
    X_combined, y_combined, test_size=0.3, random_state=42, stratify=y_combined
)

# Val/Test = 50% / 50% (of temp 30%)
X_val_unshaped, X_test_unshaped, y_val, y_test = train_test_split(
    X_temp_unshaped, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'\nTrain set size: {len(X_train_unshaped)}')
print(f'Val set size: {len(X_val_unshaped)}')
print(f'Test set size: {len(X_test_unshaped)}')

# --- Normalize Features (훈련 데이터로만 fit) --- #
# Reshape for scaler: (num_samples * window_length, num_features)
scaler = StandardScaler()

# 훈련 데이터를 reshape하여 fit
X_train_reshaped = X_train_unshaped.reshape(-1, N_FEATURES)
scaler.fit(X_train_reshaped)  # ✓ 훈련 데이터로만 fit

# 훈련, 검증, 테스트 데이터 모두에 transform 적용
X_train_scaled_reshaped = scaler.transform(X_train_reshaped)
X_val_reshaped = X_val_unshaped.reshape(-1, N_FEATURES)
X_val_scaled_reshaped = scaler.transform(X_val_reshaped)
X_test_reshaped = X_test_unshaped.reshape(-1, N_FEATURES)
X_test_scaled_reshaped = scaler.transform(X_test_reshaped)

# 원래 shape로 복원
X_train = X_train_scaled_reshaped.reshape(X_train_unshaped.shape)
X_val = X_val_scaled_reshaped.reshape(X_val_unshaped.shape)
X_test = X_test_scaled_reshaped.reshape(X_test_unshaped.shape)

Processing S2...
Processing S3...
Processing S4...
Processing S5...
Processing S6...
Processing S7...
Processing S8...
Processing S9...
Processing S10...
Processing S11...
Processing S13...
Processing S14...
Processing S15...
Processing S16...
Processing S17...

Total windows created: 33140
Feature shape: (33140, 240, 6)
Label distribution: [17605  9963  5572]

Train set size: 23198
Val set size: 4971
Test set size: 4971


## 3. PyTorch Dataset and DataLoader

In [5]:
class WesadDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = WesadDataset(X_train, y_train)
val_dataset = WesadDataset(X_val, y_val)
test_dataset = WesadDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 4. GRU Model Definition

In [6]:
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        
        # Use nn.GRU (Gated Recurrent Unit)
        # GRU is simpler and faster than LSTM but can capture similar patterns
        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True, dropout=0.3)
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # Initialize hidden state
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(device).requires_grad_()
        
        # GRU forward pass
        out, hn = self.gru(x, h0.detach())
        
        # Index hidden state of last time step
        out = self.fc(out[:, -1, :]) 
        return out

## 5. Model Training

In [7]:
model = GRUModel(N_FEATURES, HIDDEN_DIM, LAYER_DIM, OUTPUT_DIM)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Early Stopping 설정
PATIENCE = 5
MIN_DELTA = 1e-4
best_val_loss = float("inf")
epochs_no_improve = 0
best_model_state = None

print("Starting training with Early Stopping...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for i, (sequences, labels) in enumerate(train_loader):
        sequences = sequences.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * sequences.size(0)

        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_predictions / total_samples) * 100

    # Validation
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for sequences, labels in val_loader:
            sequences = sequences.to(device)
            labels = labels.to(device)

            outputs = model(sequences)
            val_loss = criterion(outputs, labels)

            val_running_loss += val_loss.item() * sequences.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / val_total
    val_epoch_acc = (val_correct / val_total) * 100

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}% | "
        f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.2f}%"
    )

    # Early Stopping 체크 (Validation Loss 기준)
    if val_epoch_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_epoch_loss
        best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        print(f"  ✅ Validation loss improved. Best Val Loss: {best_val_loss:.4f}")
    else:
        epochs_no_improve += 1
        print(f"  ⏳ No improvement for {epochs_no_improve}/{PATIENCE} epoch(s)")

        if epochs_no_improve >= PATIENCE:
            print(f"\n🛑 Early stopping triggered at epoch {epoch+1}")
            break

# best weight 복원
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.to(device)
    print(f"✅ Loaded best model weights (Best Val Loss: {best_val_loss:.4f})")

Starting training with Early Stopping...
Epoch [1/20] | Train Loss: 0.5328, Train Acc: 77.24% | Val Loss: 0.4971, Val Acc: 79.30%
  ✅ Validation loss improved. Best Val Loss: 0.4971
Epoch [2/20] | Train Loss: 0.2530, Train Acc: 90.14% | Val Loss: 0.1760, Val Acc: 93.62%
  ✅ Validation loss improved. Best Val Loss: 0.1760
Epoch [3/20] | Train Loss: 0.0996, Train Acc: 96.73% | Val Loss: 0.0466, Val Acc: 98.57%
  ✅ Validation loss improved. Best Val Loss: 0.0466
Epoch [4/20] | Train Loss: 0.0579, Train Acc: 98.12% | Val Loss: 0.1179, Val Acc: 97.12%
  ⏳ No improvement for 1/5 epoch(s)
Epoch [5/20] | Train Loss: 0.0290, Train Acc: 99.05% | Val Loss: 0.0186, Val Acc: 99.24%
  ✅ Validation loss improved. Best Val Loss: 0.0186
Epoch [6/20] | Train Loss: 0.0357, Train Acc: 98.89% | Val Loss: 0.0250, Val Acc: 99.09%
  ⏳ No improvement for 1/5 epoch(s)


KeyboardInterrupt: 

In [ ]:
# 학습 완료 후 모델 저장
SAVE_DIR = "Save_model"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "gru_model.pt")

checkpoint = {
    "model_name": "GRU",
    "model_state_dict": model.state_dict(),
    "n_features": N_FEATURES,
    "hidden_dim": HIDDEN_DIM,
    "layer_dim": LAYER_DIM,
    "output_dim": OUTPUT_DIM,
}

torch.save(checkpoint, MODEL_SAVE_PATH)
print(f"✅ Model saved to: {MODEL_SAVE_PATH}")

## 6. Model Evaluation

In [ ]:
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        outputs = model(sequences)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f'\nTest Accuracy: {accuracy:.2f} %')

# Optional: Print classification report for more details
try:
    from sklearn.metrics import classification_report
    report = classification_report(all_labels, all_preds, target_names=['baseline', 'stress', 'amusement'])
    print("\nClassification Report:")
    print(report)
except ImportError:
    print("\nPlease install scikit-learn to see the classification report: pip install -U scikit-learn")

# 각 클래스별 정확도 계산
all_labels_np = np.array(all_labels)
all_preds_np = np.array(all_preds)

class_names = ['Baseline', 'Stress', 'Amusement']
print("\n" + "="*60)
print("각 상태별 정확도")
print("="*60)

for class_idx, class_name in enumerate(class_names):
    # 해당 클래스의 샘플들
    class_mask = all_labels_np == class_idx
    class_total = class_mask.sum()
    
    if class_total > 0:
        # 해당 클래스에서 올바르게 예측된 샘플들
        class_correct = ((all_labels_np == class_idx) & (all_preds_np == class_idx)).sum()
        class_accuracy = (class_correct / class_total) * 100
        print(f"{class_name}: {class_accuracy:.2f}% ({class_correct}/{class_total})")
    else:
        print(f"{class_name}: N/A (no samples)")

print("="*60)


Test Accuracy: 99.90 %

Classification Report:
              precision    recall  f1-score   support

    baseline       1.00      1.00      1.00      2641
      stress       1.00      1.00      1.00      1494
   amusement       1.00      1.00      1.00       836

    accuracy                           1.00      4971
   macro avg       1.00      1.00      1.00      4971
weighted avg       1.00      1.00      1.00      4971


각 상태별 정확도
Baseline: 99.96% (2640/2641)
Stress: 100.00% (1494/1494)
Amusement: 99.52% (832/836)


In [ ]:
# Inference Time Benchmark (GRU)
import time
import numpy as np
import torch

if "model" not in globals():
    raise NameError("`model` 변수가 없습니다. 모델 생성/학습 셀을 먼저 실행하세요.")
if "test_loader" not in globals():
    raise NameError("`test_loader` 변수가 없습니다. DataLoader 생성 셀을 먼저 실행하세요.")

runtime_device = device if "device" in globals() else torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(runtime_device)
model.eval()


def _sync_if_cuda(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def benchmark_inference(model, data_loader, dev, warmup_steps=20):
    iterator = iter(data_loader)
    first_batch = next(iterator)
    x0 = first_batch[0].to(dev)

    for _ in range(warmup_steps):
        _ = model(x0)
    _sync_if_cuda(dev)

    batch_times = []
    total_samples = 0

    for inputs, _ in data_loader:
        inputs = inputs.to(dev)

        _sync_if_cuda(dev)
        t0 = time.perf_counter()
        _ = model(inputs)
        _sync_if_cuda(dev)
        t1 = time.perf_counter()

        batch_times.append((t1 - t0) * 1000.0)
        total_samples += inputs.size(0)

    total_time_ms = float(np.sum(batch_times))
    avg_batch_ms = float(np.mean(batch_times))
    std_batch_ms = float(np.std(batch_times))
    avg_sample_ms = total_time_ms / max(total_samples, 1)
    throughput = total_samples / (total_time_ms / 1000.0)

    return {
        "num_batches": len(batch_times),
        "num_samples": total_samples,
        "total_time_ms": total_time_ms,
        "avg_batch_ms": avg_batch_ms,
        "std_batch_ms": std_batch_ms,
        "avg_sample_ms": avg_sample_ms,
        "throughput_sps": throughput,
    }


stats = benchmark_inference(model, test_loader, runtime_device, warmup_steps=20)

print("=" * 70)
print("GRU Inference Benchmark (Test Loader)")
print("=" * 70)
print(f"Device            : {runtime_device}")
print(f"Batches           : {stats['num_batches']}")
print(f"Samples           : {stats['num_samples']}")
print(f"Total time        : {stats['total_time_ms']:.2f} ms")
print(f"Avg batch latency : {stats['avg_batch_ms']:.3f} ± {stats['std_batch_ms']:.3f} ms")
print(f"Avg sample latency: {stats['avg_sample_ms']:.4f} ms/sample")
print(f"Throughput        : {stats['throughput_sps']:.2f} samples/sec")

In [ ]:
# 실시간 환경(Batch Size 1)을 위한 전용 로더 생성
realtime_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# 기존의 벤치마크 함수를 그대로 사용하여 측정
stats_rt = benchmark_inference(model, realtime_loader, runtime_device, warmup_steps=50)

print("=" * 70)
print("실시간 응답 시간 측정 결과 (Batch Size = 1)")
print("=" * 70)
print(f"Device            : {runtime_device}")
print(f"순수 모델 응답 시간 : {stats_rt['avg_batch_ms']:.3f} ms") 
print(f"초당 처리 가능 횟수 : {stats_rt['throughput_sps']:.2f} FPS")
print("=" * 70)